# Import libraries and load data

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Path to merged_daily.csv
DATA_PATH = "../processed/merged_daily.csv"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

,date,tavg,tmin,tmax,prcp,wspd,pres,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,uv_index_max,uv_index_clear_sky_max
0,2023-01-01 00:00:00+07:00,26.5,23.0,31.0,0.6,8.9,1012.7,42.38,29.00,42.76,17.00,51.33,710.33,7.20,8.80
1,2023-01-02 00:00:00+07:00,26.7,23.0,32.0,0.0,6.3,1013.2,51.38,35.54,53.81,9.94,57.96,1090.38,7.55,8.30
2,2023-01-03 00:00:00+07:00,28.1,24.0,33.0,1.0,9.7,1012.4,40.10,27.82,39.09,5.94,54.12,662.62,7.65,8.45
3,2023-01-04 00:00:00+07:00,27.6,24.0,33.0,0.1,13.4,1011.7,28.11,19.46,28.41,13.16,46.79,495.75,6.30,9.10
4,2023-01-05 00:00:00+07:00,25.8,24.0,28.0,5.8,17.2,1011.3,16.40,11.28,21.09,11.22,51.29,352.62,3.15,9.55


# Define input and target variables

In [2]:
# Create lag-1 features for weather and air quality
for col in ["tavg", "tmin", "tmax", "prcp", "wspd", "pres",
            "pm10", "nitrogen_dioxide", "sulphur_dioxide",
            "ozone", "carbon_monoxide", "uv_index_max"]:
    df[f"{col}_lag1"] = df[col].shift(1)

# Drop the first row because lag values are missing
df_model = df.dropna().copy()

# Target variable (y): PM2.5 today
y = df_model["pm2_5"].values

# Input variables (X): all lag-1 features
feature_cols = [c for c in df_model.columns if c.endswith("_lag1")]
X = df_model[feature_cols].values

X.shape, y.shape

((364, 12), (364,))

In [3]:
df_model.head(10)

,date,tavg,tmin,tmax,prcp,wspd,pres,pm10,pm2_5,nitrogen_dioxide,...,tmax_lag1,prcp_lag1,wspd_lag1,pres_lag1,pm10_lag1,nitrogen_dioxide_lag1,sulphur_dioxide_lag1,ozone_lag1,carbon_monoxide_lag1,uv_index_max_lag1
1,2023-01-02 00:00:00+07:00,26.7,23.0,32.0,0.0,6.3,1013.2,51.38,35.54,53.81,...,31.0,0.6,8.9,1012.7,42.38,42.76,17.00,51.33,710.33,7.20
2,2023-01-03 00:00:00+07:00,28.1,24.0,33.0,1.0,9.7,1012.4,40.10,27.82,39.09,...,32.0,0.0,6.3,1013.2,51.38,53.81,9.94,57.96,1090.38,7.55
3,2023-01-04 00:00:00+07:00,27.6,24.0,33.0,0.1,13.4,1011.7,28.11,19.46,28.41,...,33.0,1.0,9.7,1012.4,40.10,39.09,5.94,54.12,662.62,7.65
4,2023-01-05 00:00:00+07:00,25.8,24.0,28.0,5.8,17.2,1011.3,16.40,11.28,21.09,...,33.0,0.1,13.4,1011.7,28.11,28.41,13.16,46.79,495.75,6.30
5,2023-01-06 00:00:00+07:00,25.7,22.0,30.0,18.3,15.2,1010.9,17.42,12.00,17.50,...,28.0,5.8,17.2,1011.3,16.40,21.09,11.22,51.29,352.62,3.15
6,2023-01-07 00:00:00+07:00,25.9,24.0,29.0,1.9,13.6,1010.3,17.44,12.11,21.02,...,30.0,18.3,15.2,1010.9,17.42,17.50,9.70,56.71,360.79,6.35
7,2023-01-08 00:00:00+07:00,25.9,23.0,29.0,4.3,6.6,1010.7,37.91,26.40,35.46,...,29.0,1.9,13.6,1010.3,17.44,21.02,13.92,61.21,543.71,4.30
8,2023-01-09 00:00:00+07:00,26.8,25.0,31.0,23.6,7.7,1011.6,52.14,36.39,26.43,...,29.0,4.3,6.6,1010.7,37.91,35.46,21.17,46.04,671.38,6.05
9,2023-01-10 00:00:00+07:00,26.3,24.0,29.0,16.2,12.2,1011.2,14.78,9.94,22.12,...,31.0,23.6,7.7,1011.6,52.14,26.43,7.73,35.67,556.00,7.05
10,2023-01-11 00:00:00+07:00,27.2,25.0,30.0,0.7,10.4,1010.7,21.29,14.02,25.90,...,29.0,16.2,12.2,1011.2,14.78,22.12,1.72,38.46,282.08,5.80


# Train / test split by time

In [4]:
n = len(df_model)
train_size = int(n * 0.8)  # First 80% for train, last 20% for test

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

len(X_train), len(X_test)

(291, 73)

# Train Linear Regression model

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate MAE, RMSE, and R^2

In [6]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f} µg/m³")
print(f"RMSE : {rmse:.2f} µg/m³")
print(f"R²   : {r2:.4f}")

MAE  : 5.12 µg/m³
RMSE : 6.41 µg/m³
R²   : 0.3595


# Inspect regression coefficients and feature importance

In [ ]:
# Retrieve regression coefficients and corresponding feature names
coefficients = model.coef_
feature_names = feature_cols  # Use feature_cols defined earlier as feature names

# Build a DataFrame to display coefficients clearly, sorted by absolute value
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients)
}).sort_values(by='Abs_Coefficient', ascending=False)
print("\nTop 10 most important features (by absolute coefficient):")

print(coef_df.head(10).to_string(index=False))


Top 10 biến quan trọng nhất (theo trị tuyệt đối):
              Feature  Coefficient  Abs_Coefficient
            tavg_lag1     0.595490         0.595490
            tmax_lag1    -0.325748         0.325748
            pm10_lag1     0.284116         0.284116
    uv_index_max_lag1    -0.272938         0.272938
nitrogen_dioxide_lag1     0.251060         0.251060
            pres_lag1     0.167903         0.167903
           ozone_lag1     0.101728         0.101728
            wspd_lag1    -0.073151         0.073151
            tmin_lag1     0.035467         0.035467
 sulphur_dioxide_lag1     0.019109         0.019109


# Compare actual vs predicted over time

In [8]:
import plotly.graph_objects as go

test_dates = df_model["date"].iloc[train_size:]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=test_dates, y=y_test,
    mode="lines", name="Actual PM2.5", line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=test_dates, y=y_pred,
    mode="lines", name="Predicted PM2.5 (Linear Regression)", line=dict(color="red", dash="dot")
))

fig.update_layout(
    title="<b>Actual vs predicted PM2.5 on test set</b>",
    xaxis_title="Date",
    yaxis_title="PM2.5 (µg/m³)",
    template="plotly_white"
)
fig.show()

In [9]:
# === COMPARISON: Linear Regression vs Baseline ===
print("\n== Linear Regression ==")
print(f"MAE  : {mae:.2f} µg/m³")
print(f"RMSE : {rmse:.2f} µg/m³")
print(f"R²   : {r2:.4f}")

# === BASELINE: predict today PM2.5 = yesterday PM2.5 (lag1) ===
df_model["pm2_5_lag1"] = df_model["pm2_5"].shift(1)

# Use the same test period
pm25_lag1_test = df_model["pm2_5_lag1"].iloc[train_size:].values
y_test_baseline = y_test  # same target as above

# Because of the shift, the first value of pm2_5_lag1 may be NaN; drop it if needed
mask = ~np.isnan(pm25_lag1_test)
pm25_lag1_test = pm25_lag1_test[mask]
y_test_baseline = y_test_baseline[mask]

mae_base = mean_absolute_error(y_test_baseline, pm25_lag1_test)
mse_base = mean_squared_error(y_test_baseline, pm25_lag1_test)
rmse_base = np.sqrt(mse_base)
r2_base = r2_score(y_test_baseline, pm25_lag1_test)

print("\n== Baseline: yesterday PM2.5 ==")
print(f"MAE  : {mae_base:.2f} µg/m³")
print(f"RMSE : {rmse_base:.2f} µg/m³")
print(f"R²   : {r2_base:.4f}")


== Linear Regression ==
MAE  : 5.12 µg/m³
RMSE : 6.41 µg/m³
R²   : 0.3595

== Baseline: yesterday PM2.5 ==
MAE  : 5.13 µg/m³
RMSE : 7.15 µg/m³
R²   : 0.2019
